## Flood Net Semantic Segmentation

This notebook processes the FloodNet dataset for semantic segmentation, training a U-Net and a PSP-Net model, and evaluating its performance. It includes steps for data loading, preparation, model training, saving, and quantify the results.

### Mount Google Drive and Extract Dataset

This cell mounts your Google Drive to access the `FloodNet.zip` file, copies it to the `/content/` directory, extracts its contents to `/content/FloodNet_local/`, and then removes the zip file.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Copy zip file to Google Colab
!cp "/content/drive/MyDrive/FloodNet Ordner/FloodNet.zip" "/content/"

# Unzip the file locally
!unzip -q /content/FloodNet.zip -d /content/FloodNet_local

# Delete zip file
!rm /content/FloodNet.zip

### Verify Dataset

This cell contains a `verify_dataset` function that checks the consistency and structure of the FloodNet dataset. It ensures that the number of images and masks in the training, validation, and test sets match, and that the filenames for images and labels correspond correctly.

In [ ]:
from glob import glob
def verify_dataset():

    # Paths for training, validation, and test sets
    dataset_dir = "/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0/"

    train_lab = glob(f"{dataset_dir}/train/train-label-img/*.png")
    train_org = glob(f"{dataset_dir}/train/train-org-img/*.jpg")
    val_lab = glob(f"{dataset_dir}/val/val-label-img/*.png")
    val_org = glob(f"{dataset_dir}/val/val-org-img/*.jpg")
    test_lab = glob(f"{dataset_dir}/test/test-label-img/*.png")
    test_org = glob(f"{dataset_dir}/test/test-org-img/*.jpg")


    # Verify the number of items in each set
    assert len(train_lab) == len(train_org) == 1445, "Training set sizes do not match."
    assert len(val_lab) == len(val_org) == 450, "Validation set sizes do not match."
    assert len(test_lab) == len(test_org) == 448, "Test set sizes do not match."

    # Verify that labels and original images match
    assert set([t.split('/')[-1].split('_')[0] for t in train_lab]) == \
           set([t.split('/')[-1].split('.')[0] for t in train_org]), "Training labels and images do not match."
    assert set([t.split('/')[-1].split('_')[0] for t in val_lab]) == \
           set([t.split('/')[-1].split('.')[0] for t in val_org]), "Validation labels and images do not match."
    assert set([t.split('/')[-1].split('_')[0] for t in test_org])== \
           set([t.split('/')[-1].split('.')[0] for t in test_org]), "Test labels and images do not match."

    print("Dataset verification passed!")


if __name__ == "__main__":
    verify_dataset()

# Dataset Provider


### Class for handling data

This cell defines the `MyFloodNetDataset` class. It handles loading image and mask pairs, resizing them to a specified size (512x512), converting them to PyTorch tensors, filtering out non-image file and do a data augmentation also a normalization.

In [ ]:
import os
import torch
import numpy as np
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from PIL import Image, ImageOps

class MyFloodNetDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_size=512, is_train=False):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.img_size = img_size
        self.is_train = is_train
        # Sort images and filter non-image files
        self.images_list = sorted([img_name for img_name in os.listdir(self.images_dir) if not img_name.startswith('.')])

    def __len__(self):
        # Get number of images
        return len(self.images_list)

    def __getitem__(self, index):
        img_name = self.images_list[index]

        # Create mask file name
        mask_name=img_name.replace(".jpg","_lab.png")

        # Join path with name
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)

        # Open images with PIL
        image = Image.open(img_path).convert("RGB")

        #turn the picture as written in the metadata (so the picture and mask has the same orientation)
        image = ImageOps.exif_transpose(image)

        #Load mask in grayscale ('L' mode) since it contains class indices, not colors
        mask = Image.open(mask_path).convert("L")

        # Resize images to 512x512
        image = transforms.Resize((self.img_size, self.img_size))(image)

        # Use NEAREST for the mask to prevent blending class indices into floating point numbers
        mask = transforms.Resize((self.img_size, self.img_size), interpolation=transforms.InterpolationMode.NEAREST)(mask)

        # Data augmentation (flip horizontally and vertical)
        if self.is_train:
            import random
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)
            if random.random() > 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask)

        # Convert to PyTorch tensors
        image = transforms.ToTensor()(image)
        mask = torch.as_tensor(np.array(mask), dtype=torch.long)

        # Apply normalization to the tensor image
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        image = normalize(image)

        return image, mask

### Data Loader Function

Here we define a get_loader helper function to streamline data loading. It links your image and mask folders to the MyFloodNetDataset and returns a PyTorch DataLoader. This DataLoader acts as an automatic pipeline, efficiently feeding well-shuffled batches of images straight to your GPU in the background during training.

In [ ]:
import torch.utils.data as td
import os

def get_loader(base_path, dataset_type, batch_size, img_size=512, is_train=False):

    # Build path for specific file
    images_dir = os.path.join(base_path, dataset_type, dataset_type + "-org-img")
    masks_dir = os.path.join(base_path, dataset_type, dataset_type + "-label-img")

    # Create dataset
    dataset = MyFloodNetDataset(images_dir=images_dir, masks_dir=masks_dir, img_size=img_size, is_train=is_train)

    return td.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        # Only shuffle during training so the model doesn't memorize the image order
        shuffle=is_train,
        # Use a couple of background CPU workers to load images while the GPU is busy
        num_workers=2,
        pin_memory=True,     # Speeds up data transfer to GPU
        prefetch_factor=2    # Preloads next batches
    )

# Model Training and Validation

### U-Net Model Training and Validation

This is the main training cell for training an U-Net Model. It defines the `train` and `validation` functions, sets up the U-Net model, Loss Function, and Adam optimizer. It then runs the training loop, including early stopping, saves model checkpoints with hyperparameters, and plots the training and validation loss history.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os
from matplotlib import pyplot as plt
from tqdm import tqdm


# Training and Validation Functions

def train(model, loss_fn, optimizer, epoch, train_ds, device):
    # Set the model to training mode
    model.train()
    running_loss = []
    # Progress bar for training
    loop = tqdm(train_ds, desc=f"Epoch {epoch} - Training")

    for (data, target) in loop:
        # Move images and masks to the active device
        data, target = data.to(device), target.to(device)

        # Reset gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        prediction = model(data.float())
        loss = loss_fn(prediction, target.long())

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track loss
        running_loss.append(loss.item())
        loop.set_postfix(loss=np.mean(running_loss))

    return np.mean(running_loss)

def validation(model, loss_fn, epoch, val_ds, device):
    # Set model to evaluation mode
    model.eval()
    running_loss = []
    loop = tqdm(val_ds, desc=f"Epoch {epoch} - Validation")

    with torch.no_grad():
        for (data, target) in loop:
            data, target = data.to(device), target.to(device)
            prediction = model(data.float())
            loss = loss_fn(prediction, target.long())
            running_loss.append(loss.item())
            loop.set_postfix(loss=np.mean(running_loss))
    return np.mean(running_loss)

# Loss Functions

class FocalLoss(nn.Module):
    """
    Focal Loss addresses class imbalance by down-weighting the loss assigned
    to well-classified examples. It forces the model to focus on hard-to-predict pixels.
    """
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma # Gamma controls how heavily easy examples are penalized
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Calculate standard Cross-Entropy without reducing it to a single number
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        # pt represents the model's predicted probability for the TRUE class
        pt = torch.exp(-ce_loss)

        # Apply the focal loss formula: (1 - pt)^gamma acts as the modulating factor
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class DiceLoss(nn.Module):
    """
    Dice Loss measures the spatial overlap (Intersection over Union) between
    the prediction and the ground truth. Highly effective for segmentation tasks.
    """
    def __init__(self, num_classes=10, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.num_classes = num_classes
        # Smooth prevents division by zero if both prediction and target are empty
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Convert raw logits to probabilities
        probs = F.softmax(inputs, dim=1)


        # Original shape: [B, H, W, Classes], we permute to match inputs: [B, Classes, H, W]
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        # Sum over spatial dimensions (Batch, Height, Width), keeping the Class dimension
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dim=dims)
        cardinality = torch.sum(probs + targets_one_hot, dim=dims)

        # Calculate the Dice Coefficient (Score)
        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)

        # Loss is 1 - Score. We return the average loss across all classes.
        return 1 - dice_score.mean()


class HybridFocalDiceLoss(nn.Module):
    """
    Combines Focal Loss (for class imbalance) and Dice Loss (for spatial accuracy).
    This generally yields better borders and captures rare classes efficiently.
    """
    def __init__(self, num_classes=10, alpha=1, gamma=2, w_focal=1.0, w_dice=1.0):
        super(HybridFocalDiceLoss, self).__init__()
        self.focal = FocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss(num_classes=num_classes)
        # Weights allow tuning the impact of each respective loss
        self.w_focal = w_focal
        self.w_dice = w_dice

    def forward(self, inputs, targets):
        # Compute and combine both losses
        return self.w_focal * self.focal(inputs, targets) + self.w_dice * self.dice(inputs, targets)

# Setup & Execution

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#Load a standard U-Net architecture
model = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet',
                       in_channels=3, out_channels=10, init_features=64, pretrained=False).to(device)

# Initialize Loss
loss_fn = HybridFocalDiceLoss(num_classes=10)

# Setup the optimizers
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Hyperparameters
epochs = 40
patience = 6


if __name__ == '__main__':
    # Configuration
    model_save_path = "/content/drive/MyDrive/FloodNet_Saved_Training/"
    os.makedirs(model_save_path, exist_ok=True)

    # Variables to track our progress
    all_tr_losses, all_val_losses = [], []
    best_val_loss = float('inf')
    count = 0
    start_epoch = 1

    # Spin up our DataLoaders
    train_ds = get_loader(base_path="/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0/", dataset_type="train", batch_size=4, is_train=True)
    val_ds = get_loader(base_path="/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0/", dataset_type="val", batch_size=4, is_train=False)

    # Checkpointing Logic (Resume Training)
    checkpoint_path = os.path.join(model_save_path, "best_unet_model_fnloss_epoch_10.pt")

    # If the notebook disconnected, we don't want to start from scratch.
    # This block loads the last saved state and resumes exactly where it left off.
    if os.path.exists(checkpoint_path):
        print(f"Found Checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Restore weights, optimizer state, and loss history
        model.load_state_dict(checkpoint['net_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])

        all_tr_losses = checkpoint.get('all_tr_losses', [])
        all_val_losses = checkpoint.get('all_val_losses', [])
        best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        count = checkpoint.get('count', 0)
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming training from epoch {start_epoch}.")

    # Training Loop
    for epoch in range(start_epoch, epochs + 1):
        tr_loss = train(model, loss_fn, optimizer, epoch, train_ds, device)
        val_loss = validation(model, loss_fn, epoch, val_ds, device)

        all_tr_losses.append(tr_loss)
        all_val_losses.append(val_loss)

        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"-> New best Validation Loss! Saving model state.")
            torch.save({
                "epoch": epoch,
                "net_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "count": count,
                "all_tr_losses": all_tr_losses,
                "all_val_losses": all_val_losses
            }, os.path.join(model_save_path, f"best_unet_model_fnloss_epoch_{epoch}.pt"))

        # Early Stopping: If the model hasn't improved after 'patience' epochs, stop training
        # to prevent it from just memorizing the training data (overfitting).
        if len(all_val_losses) > 1:
            if val_loss > all_val_losses[-2]:
                count += 1
                print(f"-> Validation Loss increased! Patience: {count}/{patience}")
            else:
                count = 0
        if count >= patience:
            print(f"\nEarly Stopping at epoch {epoch}!")
            break

    # Plotting Results
    plt.figure(figsize=(10, 5))
    plt.plot(all_tr_losses, label="Train Loss")
    plt.plot(all_val_losses, label="Val Loss")
    plt.title("U-Net Training History")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

### PSP Net Model Training and Validation

This is the main training cell for training the PSP-Net model. It defines the train and validation functions, sets up the PSP-Net model, Loss Function, and Adam optimizer. It then runs the training loop, including early stopping, saves model checkpoints with hyperparameters, and plots the training and validation loss history

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os
import segmentation_models_pytorch as smp
from matplotlib import pyplot as plt
from tqdm import tqdm
from torch.optim.lr_scheduler import PolynomialLR

# Training and Validation Functions

def train(model, loss_fn, optimizer, epoch, train_ds, device):
    # Set the model to training mode
    model.train()
    running_loss = []
    # Progress bar for training
    loop = tqdm(train_ds, desc=f"Epoch {epoch} - Training")

    for (data, target) in loop:
        # Move images and masks to the active device
        data, target = data.to(device), target.to(device)

        # Reset gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        prediction = model(data.float())
        loss = loss_fn(prediction, target.long())

        # Backward pass
        loss.backward()
        optimizer.step()

        # Track loss
        running_loss.append(loss.item())
        loop.set_postfix(loss=np.mean(running_loss))
    return np.mean(running_loss)

def validation(model, loss_fn, epoch, val_ds, device):
    # Set model to evaluation mode
    model.eval()
    running_loss = []
    loop = tqdm(val_ds, desc=f"Epoch {epoch} - Validation")

    with torch.no_grad():
        for (data, target) in loop:
            data, target = data.to(device), target.to(device)
            prediction = model(data.float())
            loss = loss_fn(prediction, target.long())
            running_loss.append(loss.item())
            loop.set_postfix(loss=np.mean(running_loss))
    return np.mean(running_loss)

# Loss Functions

class FocalLoss(nn.Module):
    """
    Focal Loss addresses class imbalance by down-weighting the loss assigned
    to well-classified examples. It forces the model to focus on hard-to-predict pixels.
    """
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma # Gamma controls how heavily easy examples are penalized
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Calculate standard Cross-Entropy without reducing it to a single number
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        # pt represents the model's predicted probability for the TRUE class
        pt = torch.exp(-ce_loss)

        # Apply the focal loss formula: (1 - pt)^gamma acts as the modulating factor
        focal_loss = self.alpha * ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class DiceLoss(nn.Module):
    """
    Dice Loss measures the spatial overlap (Intersection over Union) between
    the prediction and the ground truth. Highly effective for segmentation tasks.
    """
    def __init__(self, num_classes=10, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.num_classes = num_classes
        # Smooth prevents division by zero if both prediction and target are empty
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Convert raw logits to probabilities
        probs = F.softmax(inputs, dim=1)

        # Original shape: [B, H, W, Classes], we permute to match inputs: [B, Classes, H, W]
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        # Sum over spatial dimensions (Batch, Height, Width), keeping the Class dimension
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dim=dims)
        cardinality = torch.sum(probs + targets_one_hot, dim=dims)

        # Calculate the Dice Coefficient (Score)
        dice_score = (2. * intersection + self.smooth) / (cardinality + self.smooth)

        # Loss is 1 - Score. We return the average loss across all classes.
        return 1 - dice_score.mean()


class HybridFocalDiceLoss(nn.Module):
    """
    Combines Focal Loss (for class imbalance) and Dice Loss (for spatial accuracy).
    This generally yields better borders and captures rare classes efficiently.
    """
    def __init__(self, num_classes=10, alpha=1, gamma=2, w_focal=1.0, w_dice=1.0):
        super(HybridFocalDiceLoss, self).__init__()
        self.focal = FocalLoss(alpha=alpha, gamma=gamma)
        self.dice = DiceLoss(num_classes=num_classes)
        # Weights allow tuning the impact of each respective loss
        self.w_focal = w_focal
        self.w_dice = w_dice

    def forward(self, inputs, targets):
        # Compute and combine both losses
        return self.w_focal * self.focal(inputs, targets) + self.w_dice * self.dice(inputs, targets)



# Model Setup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize PSPNet
model = smp.PSPNet(
    encoder_name="resnet101",
    encoder_weights=None,
    in_channels=3,
    classes=10
).to(device)

#Initialize optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Initialise Loss und Optimizer
loss_fn = HybridFocalDiceLoss(num_classes=10)

# Hyperparameters
epochs = 40
patience = 6

if __name__ == '__main__':
    model_save_path = "/content/drive/MyDrive/FloodNet_Saved_Training/"
    os.makedirs(model_save_path, exist_ok=True)

    # Tracking variables
    all_tr_losses, all_val_losses = [], []
    best_val_loss = float('inf')
    count = 0
    start_epoch = 1

    # Spin up DataLoaders
    train_ds = get_loader(base_path="/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0/", dataset_type="train", batch_size=4, is_train=True)
    val_ds = get_loader(base_path="/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0/", dataset_type="val", batch_size=4, is_train=False)

    # Checkpointing Logic (Resume Training)
    checkpoint_path = os.path.join(model_save_path, "best_pspnet_model_fnloss_resnet101_epoch_38.pt")

    if os.path.exists(checkpoint_path):
        print(f"Found Checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

        # Restore weights, optimizer state, and loss history
        model.load_state_dict(checkpoint['net_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])

        all_tr_losses = checkpoint.get('all_tr_losses', [])
        all_val_losses = checkpoint.get('all_val_losses', [])
        best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        count = checkpoint.get('count', 0)
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resuming training from epoch {start_epoch}.")

    # Training Loop
    for epoch in range(start_epoch, epochs + 1):
        tr_loss = train(model, loss_fn, optimizer, epoch, train_ds, device)
        val_loss = validation(model, loss_fn, epoch, val_ds, device)

        all_tr_losses.append(tr_loss)
        all_val_losses.append(val_loss)


        # Save checkpoint if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"-> New best Validation Loss! Saving model state.")
            torch.save({
                "epoch": epoch,
                "net_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "count": count,
                "all_tr_losses": all_tr_losses,
                "all_val_losses": all_val_losses
            }, os.path.join(model_save_path, f"best_pspnet_model_fnloss_resnet101_epoch_{epoch}.pt"))

        # Early Stopping: If the model hasn't improved after 'patience' epochs, stop training
        # to prevent it from just memorizing the training data (overfitting).
        if len(all_val_losses) > 1:
            if val_loss > all_val_losses[-2]:
                count += 1
                print(f"-> Validation Loss increased! Patience: {count}/{patience}")
            else:
                count = 0
        if count >= patience:
            print(f"\nEarly Stopping at epoch {epoch}!")
            break

    # Plotting Results
    plt.figure(figsize=(10, 5))
    plt.plot(all_tr_losses, label="Train Loss")
    plt.plot(all_val_losses, label="Val Loss")
    plt.title("PSP Net Training History")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.show()

### Visualizing the Training Progress
This script helps you evaluate your model's learning curve. Instead of starting from scratch, it automatically hunts down your most recently saved checkpoint, extracts the stored training and validation loss data, and plots a clean, high-resolution graph.


In [ ]:
import os
import matplotlib.pyplot as plt
import torch

# define path
model_save_path = "/content/drive/MyDrive/FloodNet_Saved_Training"
# path to save plot
plot_save_path = os.path.join(model_save_path, "loss_plot_pspnet_model_fnloss_resnet101_epoch_38.png")

# find the newest model
checkpoints = [f for f in os.listdir(model_save_path) if f.startswith("best_pspnet_model_fnloss_epoch_") and f.endswith(".pt")]

if checkpoints:
    epochs_saved = [int(f.split("_")[-1].split(".pt")[0]) for f in checkpoints]
    latest_epoch = max(epochs_saved)
    latest_checkpoint_path = os.path.join(model_save_path, f"best_pspnet_model_fnloss_epoch_{latest_epoch}.pt")

    print(f"Load loss history from: {latest_checkpoint_path}")

    # load checkpoint
    checkpoint = torch.load(latest_checkpoint_path, map_location='cpu', weights_only=False)

    # extract the loss lists
    all_tr_losses = checkpoint.get('all_tr_losses', [])
    all_val_losses = checkpoint.get('all_val_losses', [])

    if len(all_tr_losses) == 0 or len(all_val_losses) == 0:
        print("Warnung: In diesem Checkpoint wurden keine Loss-Listen gefunden!")
    else:
        # plot figure
        plt.figure(figsize=(10, 6))
        plt.plot(all_tr_losses, label='Training Loss', color='blue', linestyle='-')
        plt.plot(all_val_losses, label='Validation Loss', color='orange', linestyle='-')

        plt.title(f'Training & Validation Loss (Epoch {latest_epoch})')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)

        # save
        plt.savefig(plot_save_path, bbox_inches='tight', dpi=300)
        plt.show()

        print(f"✓ Diagram saved under: {plot_save_path}")
else:
    print("No Checkpoints found")

### Generating a PDF Report about the test images and model output

This cell scales up the inference process by running our trained Model on every single image in the test set. It compiles the original images, ground truth masks, and model predictions into a clean, multi-page PDF report—complete with a color-coded legend.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os
import torchvision.transforms as T
import segmentation_models_pytorch as smp
from PIL import Image, ImageOps
from matplotlib.backends.backend_pdf import PdfPages


# configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = "/content/drive/MyDrive/FloodNet_Saved_Training/best_pspnet_model_fnloss_resnet101_epoch_38.pt"
BASE_PATH = "/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0"
output_pdf = "/content/drive/MyDrive/FloodNet_Predictions_Report_pspnet_model_fnloss_resnet101.pdf"

# load model (U-Net or PSP-Net depending)
model = smp.PSPNet(
    encoder_name="resnet101",        # The backbone (feature extractor)
    encoder_weights=None,
    in_channels=3,                  # RGB images (FloodNet)
    classes=10                      # Number of classes in your FloodNet dataset
).to(device)

#load the saved dictionary into the model
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['net_state_dict'])
model.eval()

# Dataset Preperation
img_dir = os.path.join(BASE_PATH, "test", "test-org-img")
mask_dir = os.path.join(BASE_PATH, "test", "test-label-img")
image_filenames = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

target_size = (512, 512)


transform_img = T.Compose([
    T.Resize(target_size),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Labels for legend
class_names = ['Background', 'Building-flooded', 'Building-non-flooded', 'Road-flooded',
               'Road-non-flooded', 'Water', 'Tree', 'Vehicle', 'Pool', 'Grass']

# create Report
with PdfPages(output_pdf) as pdf:
    for filename in image_filenames:
        print(f"Processing: {filename}")
        img_path = os.path.join(img_dir, filename)
        mask_path = os.path.join(mask_dir, filename.replace(".jpg", "_lab.png"))

        # Load and align the input image
        img_pil = Image.open(img_path).convert("RGB")
        img_pil = ImageOps.exif_transpose(img_pil)
        img_tensor = transform_img(img_pil).unsqueeze(0).to(device)

        # load ground truth mask
        has_mask = os.path.exists(mask_path)
        if has_mask:
          mask_pil = Image.open(mask_path)
          mask_pil = ImageOps.exif_transpose(mask_pil)
          mask_tensor = torch.as_tensor(np.array(mask_pil.resize(target_size, Image.NEAREST)), dtype=torch.long)
        else:
          print(f"Error: No mask found.")
          mask_tensor = torch.zeros(target_size, dtype=torch.long)

        #Run inference without tracking gradients
        with torch.no_grad():
            raw_prediction = model(img_tensor)
            predicted_mask = torch.argmax(raw_prediction, dim=1).squeeze(0)


        # Plot
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        # Denormalize the tensor so it displays correctly as an RGB image
        img_plot = img_tensor.squeeze(0).cpu().permute(1, 2, 0).numpy()
        img_plot = (std := np.array([0.229, 0.224, 0.225])) * img_plot + (mean := np.array([0.485, 0.456, 0.406]))
        img_plot = np.clip(img_plot, 0, 1)

        # Set headers
        axs[0].set_title(f"1. Original {filename}", fontsize=14, fontweight="bold")
        axs[1].set_title("2. Ground Truth Mask", fontsize=14, fontweight="bold")
        axs[2].set_title("3. Model Prediction", fontsize=14, fontweight="bold")

        #display images and turn off axis grids
        axs[0].imshow(img_plot)
        axs[0].axis('off')
        axs[1].imshow(mask_tensor.cpu().numpy(), cmap='tab10', vmin=0, vmax=9)
        axs[1].axis('off')
        axs[2].imshow(predicted_mask.cpu().numpy(), cmap='tab10', vmin=0, vmax=9)
        axs[2].axis('off')

        im = axs[2].imshow(predicted_mask.cpu().numpy(), cmap='tab10', vmin=0, vmax=9)
        axs[2].set_title("Prediction")
        axs[2].axis('off')

        # plot legend
        colors = [im.cmap(im.norm(value)) for value in range(10)]
        patches = [plt.Rectangle((0,0),1,1, color=colors[i]) for i in range(len(class_names))]

        axs[2].legend(patches, class_names, loc='upper left', bbox_to_anchor=(1.05, 1), fontsize='small')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pdf.savefig(fig)
        plt.close()

print(f"PDF saved under {output_pdf}")

### Final Model Evaluation
 This cell runs your fully trained model across the entire test dataset to calculate its accuracies. It builds one Confusion Matrix for the whole dataset. From this, it extracts the true global Mean IoU, Dice Score, and overall Pixel Accuracy.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from sklearn.metrics import confusion_matrix
import seaborn as sns
from PIL import Image, ImageOps
import segmentation_models_pytorch as smp

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = "/content/drive/MyDrive/FloodNet_Saved_Training/best_unet_model_final_epoch_40.pt"
BASE_PATH = "/content/FloodNet_local/FloodNet/FloodNet-Supervised_v1.0"
test_img_dir = os.path.join(BASE_PATH, "test", "test-org-img")
test_mask_dir = os.path.join(BASE_PATH, "test", "test-label-img")
target_size = (512, 512)
num_classes = 10

class_names = {
    0: 'Background',
    1: 'Building (Flooded)',
    2: 'Building (Non-Flooded)',
    3: 'Road (Flooded)',
    4: 'Road (Non-Flooded)',
    5: 'Water',
    6: 'Tree',
    7: 'Vehicle',
    8: 'Pool',
    9: 'Grass'
}

# load Model (U-Net or PSP-Net)
model = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet', in_channels=3, out_channels=10, init_features=64, pretrained=False).to(device)

if os.path.exists(checkpoint_path):
  #load the trained weights
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['net_state_dict'])
    model.eval()
else:
    print("Milestone not found")
    exit()

# empty lists to save metrics
all_ious = []
all_dices = []
all_pixel_accuracies = []
total_confusion_matrix = np.zeros((num_classes, num_classes))

image_files = [f for f in os.listdir(test_img_dir) if f.endswith(".jpg")]
print(f"Start Evaluation over {len(image_files)} pictures...")

# for loop for every test image
with torch.no_grad():
    for img_filename in image_files:
        img_path = os.path.join(test_img_dir, img_filename)
        mask_path = os.path.join(test_mask_dir, img_filename.replace(".jpg", "_lab.png"))

        if not os.path.exists(mask_path): continue

        # Load and align image
        img_pil = Image.open(img_path).convert("RGB")
        img_pil = ImageOps.exif_transpose(img_pil)

        # Load and align mask
        mask_pil = Image.open(mask_path)

        # transform inputs to match training configuration
        transform = T.Compose([T.Resize(target_size), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        mask_tensor = torch.as_tensor(np.array(TF.resize(mask_pil, target_size, interpolation=T.InterpolationMode.NEAREST)), dtype=torch.long).to(device)

        # Prediction
        raw_pred = model(img_tensor)
        pred_mask = torch.argmax(raw_pred, dim=1).squeeze(0)

        # Flatten arrays to easily compare pixels one-by-one
        flat_pred = pred_mask.cpu().numpy().flatten()
        flat_target = mask_tensor.cpu().numpy().flatten()

        # Track pixel accuracy for this image
        all_pixel_accuracies.append((flat_pred == flat_target).mean())

        # Accumulate data into the global confusion matrix
        total_confusion_matrix += confusion_matrix(flat_target, flat_pred, labels=range(num_classes))

# Calculate Global Metrics

# True Positives (Intersection) are the values on the diagonal of the matrix
intersection = np.diag(total_confusion_matrix)

# Total Ground Truth Pixels per class (Row sum)
ground_truth_set = total_confusion_matrix.sum(axis=1)

# Total Predicted Pixels per class (Column sum)
predicted_set = total_confusion_matrix.sum(axis=0)

union = ground_truth_set + predicted_set - intersection

# Calculate IoU per class
# To avoid dividing by zero, we ignore classes that don't exist in the test set
valid_classes = union > 0
class_ious = np.zeros(num_classes)
class_ious[valid_classes] = intersection[valid_classes] / union[valid_classes]

# Calculate Dice Score per class
class_dices = np.zeros(num_classes)
valid_dice_classes = (ground_truth_set + predicted_set) > 0
class_dices[valid_dice_classes] = (2 * intersection[valid_dice_classes]) / (ground_truth_set[valid_dice_classes] + predicted_set[valid_dice_classes])

# Calculate Mean IoU and Mean Dice (only averaging over valid classes)
mean_iou = np.mean(class_ious[valid_classes])
mean_dice = np.mean(class_dices[valid_dice_classes])
overall_pixel_acc = np.mean(all_pixel_accuracies)

# save
save_dir = os.path.dirname(checkpoint_path)
metrics_file_path = os.path.join(save_dir, "evaluation_metrics_unet_model_final_epoch_40.txt")
plot_file_path = os.path.join(save_dir, "confusion_matrix_unet_model_fnloss_epoch_40.png")
matrix_raw_path = os.path.join(save_dir, "confusion_matrix_raw_unet_model_fnloss_resnet101_epoch_40.npy")

with open(metrics_file_path, "w") as f:
    f.write("--- Accuracy Assessment ---\n")
    f.write(f"Mean IoU (mIoU): {mean_iou:.4f}\n")
    f.write(f"Mean Dice Score: {mean_dice:.4f}\n")
    f.write(f"Overall Pixel Accuracy: {overall_pixel_acc:.4f}\n\n")

    f.write("Class-specific IoU:\n")
    for i in range(num_classes):
        f.write(f"{class_names[i]}: {class_ious[i]:.4f}\n")

# print to console
print(f"\n✓ Metrics saved under: {metrics_file_path}\n")
print("--- Accuracy Assessment ---")
print(f"Mean IoU (mIoU): {mean_iou:.4f}")
print(f"Mean Dice Score: {mean_dice:.4f}")
print(f"Overall Pixel Accuracy: {overall_pixel_acc:.4f}")

print("\nClass-specific IoU:")
for i in range(num_classes):
    print(f"{class_names[i]}: {class_ious[i]:.4f}")

plt.figure(figsize=(12, 10))
sns.heatmap(total_confusion_matrix.astype(int), annot=True, fmt='d', cmap='Blues',
            xticklabels=list(class_names.values()),
            yticklabels=list(class_names.values()))
plt.title("Confusion Matrix")
plt.xlabel("Prediction")
plt.ylabel("Ground Truth")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

# save confusion matrix
plt.savefig(plot_file_path, bbox_inches='tight', dpi=300)
print(f"✓ Confusion Matrix Plot saved under: {plot_file_path}")

plt.show()

# save also the raw numbers
np.save(matrix_raw_path, total_confusion_matrix)
print(f"✓ Confusion Matrix Data saved under: {matrix_raw_path}")